## 1. Configuration & Setup

In [2]:
import os
import hashlib
import json
import re
import requests
import pandas as pd
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
import logging
from dotenv import load_dotenv

BASE_DIR = Path("..").resolve()
# Load environment variables from .env file
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG = {
    "REPORTS_CSV": BASE_DIR / "data" / "data_source" / "reports.csv",
    "AI_SYS_TAXONOMY": BASE_DIR / "data" / "taxonomy" / "ai_sys_taxonomy.txt",
    "AI_RISK_TAXONOMY": BASE_DIR / "data" / "taxonomy" / "ai_risk_taxonomy.txt",
    "OUTPUT_JSON": BASE_DIR / "data" / "outputs" / "output_reprots.json",

    "OLLAMA_ENDPOINT": os.getenv("OLLAMA_ENDPOINT"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),

    "USE_CACHE": True,
    "CACHE_DIR": BASE_DIR / "data" / "outputs" / ".cache" / "risk_sources_preprocess_refactor",

    "MAX_TEXT_CHARS": 8000,
    "MAX_ROWS": 100,

    "SCOPE_TASK": "Information Retrieval",
}

if not CONFIG["OLLAMA_ENDPOINT"]:
    raise ValueError("OLLAMA_ENDPOINT is not set in .env")
if not CONFIG["OLLAMA_MODEL"]:
    raise ValueError("OLLAMA_MODEL is not set in .env")

# Create output directory
Path(CONFIG["CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["OUTPUT_JSON"]).parent.mkdir(parents=True, exist_ok=True)

# Initialize LLM cache
Path(CONFIG["CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
LLM_CACHE = {}  # optional

def cache_key(*parts):
    """Generate cache key from parts"""
    text = "||".join([re.sub(r"\s+", " ", str(p).strip().lower()) for p in parts])
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def cache_get(stage: str, key: str) -> Optional[dict]:
    if not CONFIG["USE_CACHE"]:
        return None

    path = Path(CONFIG["CACHE_DIR"]) / f"{stage}_{key}.json"
    if not path.exists():
        return None

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def cache_set(stage: str, key: str, value: dict) -> None:
    if not CONFIG["USE_CACHE"]:
        return

    cache_dir = Path(CONFIG["CACHE_DIR"])
    cache_dir.mkdir(parents=True, exist_ok=True)

    path = cache_dir / f"{stage}_{key}.json"
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

logger.info("✓ Configuration loaded successfully")
logger.info(f"  Reports CSV: {CONFIG['REPORTS_CSV']}")
logger.info(f"  System taxonomy: {CONFIG['AI_SYS_TAXONOMY']}")
logger.info(f"  Risk taxonomy: {CONFIG['AI_RISK_TAXONOMY']}")
logger.info(f"  Output file: {CONFIG['OUTPUT_JSON']}")
logger.info(f"  LLM Model: {CONFIG['OLLAMA_MODEL']}")
logger.info(f"  Target task: {CONFIG['SCOPE_TASK']}")
logger.info(f"  Max rows: {CONFIG['MAX_ROWS'] or 'ALL'}")

2026-04-08 04:58:00,300 - __main__ - INFO - ✓ Configuration loaded successfully
2026-04-08 04:58:00,302 - __main__ - INFO -   Reports CSV: D:\Dev\airiskkg\data\data_source\reports.csv
2026-04-08 04:58:00,303 - __main__ - INFO -   System taxonomy: D:\Dev\airiskkg\data\taxonomy\ai_sys_taxonomy.txt
2026-04-08 04:58:00,304 - __main__ - INFO -   Risk taxonomy: D:\Dev\airiskkg\data\taxonomy\ai_risk_taxonomy.txt
2026-04-08 04:58:00,305 - __main__ - INFO -   Output file: D:\Dev\airiskkg\data\outputs\output_reprots.json
2026-04-08 04:58:00,306 - __main__ - INFO -   LLM Model: llama3.1:latest
2026-04-08 04:58:00,307 - __main__ - INFO -   Target task: Information Retrieval
2026-04-08 04:58:00,308 - __main__ - INFO -   Max rows: 100


## 2. Helpers: cache, parsing, and LLM calls

In [3]:

def extract_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, flags=re.DOTALL)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except Exception:
            pass

    first_obj = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if first_obj:
        try:
            return json.loads(first_obj.group(1))
        except Exception:
            pass

    return None


def ollama_generate(prompt: str, model: Optional[str] = None, endpoint: Optional[str] = None, **params) -> str:
    url = f"{(endpoint or CONFIG['OLLAMA_ENDPOINT']).rstrip('/')}/api/generate"
    payload = {
        "model": model or CONFIG["OLLAMA_MODEL"],
        "prompt": prompt,
        "stream": False,
    }
    payload.update(params)

    response = requests.post(url, json=payload, timeout=300)
    response.raise_for_status()
    data = response.json()

    if "response" in data:
        return data["response"].strip()
    if "text" in data:
        return data["text"].strip()
    return ""


def llm_json(stage: str, prompt: str, cache_id: str) -> dict:
    key = cache_key(stage, cache_id, prompt[:2000])

    cached = cache_get(stage, key)
    if cached is not None:
        return cached if isinstance(cached, dict) else {
            "_parse_error": True,
            "_raw_text": str(cached)[:5000]
        }

    raw = ollama_generate(prompt)
    parsed = extract_json_object(raw)

    if parsed is None or not isinstance(parsed, dict):
        parsed = {
            "_parse_error": True,
            "_raw_text": raw[:5000] if isinstance(raw, str) else str(raw)[:5000]
        }

    cache_set(stage, key, parsed)
    return parsed


def ollama_generate(prompt: str, model: Optional[str] = None, endpoint: Optional[str] = None, **params) -> str:
    url = f"{(endpoint or CONFIG['OLLAMA_ENDPOINT']).rstrip('/')}/api/generate"
    payload = {
        "model": model or CONFIG["OLLAMA_MODEL"],
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0
        }
    }

    if params:
        payload["options"].update(params)

    response = requests.post(url, json=payload, timeout=300)
    response.raise_for_status()
    data = response.json()

    if "response" in data:
        return data["response"].strip()
    if "text" in data:
        return data["text"].strip()
    return ""

    cache_set(stage, key, parsed)
    return parsed


## 3. Load taxonomies and incident reports

In [4]:

def read_text(path_str: str) -> str:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path.read_text(encoding="utf-8")


ai_sys_taxonomy_text = read_text(CONFIG["AI_SYS_TAXONOMY"])
ai_risk_taxonomy_text = read_text(CONFIG["AI_RISK_TAXONOMY"])

reports_df = pd.read_csv(CONFIG["REPORTS_CSV"])

print(f"Loaded system taxonomy: {len(ai_sys_taxonomy_text)} chars")
print(f"Loaded risk taxonomy:   {len(ai_risk_taxonomy_text)} chars")
print(f"Loaded reports:         {len(reports_df)} rows")

reports_df.head(3)


Loaded system taxonomy: 4576 chars
Loaded risk taxonomy:   2734 chars
Loaded reports:         6642 rows


,_id,authors,date_downloaded,date_modified,date_published,date_submitted,description,epoch_date_downloaded,epoch_date_modified,epoch_date_published,...,image_url,language,ref_number,report_number,source_domain,submitters,text,title,url,tags
0,5d34b8c29ced494f010ed45a,Alistair Barr,2019-04-13T00:00:00Z,2020-06-14T00:00:00Z,2015-05-19T00:00:00Z,2019-06-01T00:00:00Z,Child and consumer advocacy groups complained ...,NaN,NaN,1431993600,...,http://si.wsj.net/public/resources/images/BN-I...,en,NaN,1,blogs.wsj.com,Roman Yampolskiy,Child and consumer advocacy groups complained ...,Google’s YouTube Kids App Criticized for ‘Inap...,https://blogs.wsj.com/digits/2015/05/19/google...,NaN
1,5d34b8c29ced494f010ed461,Sapna Maheshwari,2019-04-13T00:00:00Z,2020-06-14T00:00:00Z,2018-04-26T00:00:00Z,2019-06-01T00:00:00Z,Parents will be able to handpick the channels ...,NaN,NaN,1524700800,...,https://static01.nyt.com/images/2017/11/07/bus...,en,NaN,8,nytimes.com,Roman Yampolskiy,"YouTube Kids, which has been criticized for in...","YouTube Kids, Criticized for Content, Introduc...",https://www.nytimes.com/2018/04/25/business/me...,NaN
2,5d34b8c29ced494f010ed464,K.G Orphanides,2019-04-13T00:00:00Z,2020-06-14T00:00:00Z,2018-03-23T00:00:00Z,2019-06-01T00:00:00Z,Children's search terms on YouTube are still a...,NaN,NaN,1521763200,...,https://wi-images.condecdn.net/image/ye8GWkPPM...,en,NaN,11,wired.co.uk,Roman Yampolskiy,Video still of a reproduced version of Minnie ...,Children's YouTube is still churning out blood...,https://www.wired.co.uk/article/youtube-for-ki...,NaN


## 4. Normalize report records

In [5]:

TEXT_CANDIDATE_COLUMNS = ["title", "description", "text", "summary", "content", "body"]
ID_CANDIDATE_COLUMNS = ["report_id", "incident_id", "id", "_id", "report_number"]
URL_CANDIDATE_COLUMNS = ["url", "source_url", "link"]
SOURCE_CANDIDATE_COLUMNS = ["source_domain", "source", "publisher"]


def first_existing_value(row: pd.Series, columns: List[str], default: str = "") -> str:
    for col in columns:
        if col in row.index and pd.notna(row[col]):
            value = str(row[col]).strip()
            if value:
                return value
    return default


def normalize_report_row(row: pd.Series) -> Dict[str, Any]:
    record_id = first_existing_value(row, ID_CANDIDATE_COLUMNS, default="")
    if not record_id:
        record_id = f"row_{row.name}"

    title = first_existing_value(row, ["title"], default="")
    description = first_existing_value(row, ["description", "summary"], default="")
    source_domain = first_existing_value(row, SOURCE_CANDIDATE_COLUMNS, default="")
    url = first_existing_value(row, URL_CANDIDATE_COLUMNS, default="")

    text_parts = []
    for col in TEXT_CANDIDATE_COLUMNS:
        if col in row.index and pd.notna(row[col]):
            val = str(row[col]).strip()
            if val:
                text_parts.append(f"{col.upper()}: {val}")

    full_text = "\n\n".join(text_parts)
    full_text = full_text[:CONFIG["MAX_TEXT_CHARS"]]

    return {
        "report_id": record_id,
        "title": title,
        "description": description,
        "source_domain": source_domain,
        "url": url,
        "full_text": full_text,
    }


normalized_preview = [normalize_report_row(reports_df.iloc[i]) for i in range(min(2, len(reports_df)))]
normalized_preview


[{'report_id': '5d34b8c29ced494f010ed45a',
  'title': 'Google’s YouTube Kids App Criticized for ‘Inappropriate Content’',
  'description': 'Child and consumer advocacy groups complained to the FTC that Google’s new YouTube Kids app contains “inappropriate content,” including explicit sexual language and jokes about pedophilia.',
  'source_domain': 'blogs.wsj.com',
  'url': 'https://blogs.wsj.com/digits/2015/05/19/googles-youtube-kids-app-criticized-for-inappropriate-content/',
  'full_text': 'TITLE: Google’s YouTube Kids App Criticized for ‘Inappropriate Content’\n\nDESCRIPTION: Child and consumer advocacy groups complained to the FTC that Google’s new YouTube Kids app contains “inappropriate content,” including explicit sexual language and jokes about pedophilia.\n\nTEXT: Child and consumer advocacy groups complained to the Federal Trade Commission Tuesday that Google’s new YouTube Kids app contains “inappropriate content,” including explicit sexual language and jokes about pedophilia

## 5. Relevance gate: keep retrieval-oriented incidents only

In [6]:

IR_HINTS = [
    # retrieval / search
    "information retrieval",
    "search",
    "search engine",
    "retrieval",
    "retrieve",
    "document retrieval",
    "semantic search",
    "knowledge retrieval",
    "knowledge base",
    "vector search",
    "search results",
    "top results",

    # QA / assistant
    "question answering",
    "qa",
    "answer engine",
    "chatbot",
    "assistant",
    "virtual assistant",
    "knowledge assistant",
    "rag",
    "retrieval augmented generation",
    "retrieval-augmented",
    "answer",

    # ranking / recommendation / surfacing
    "ranking",
    "ranked",
    "recommendation",
    "recommended",
    "recommender",
    "suggested",
    "personalized results",
    "algorithmic recommendation",
    "content surfaced",
    "content ranking",
    "feed",
    "news feed",

    # information selection / surfacing
    "surfaced",
    "prioritized",
    "recommended content",
    "shown to users",
    "displayed results"
]


def cheap_ir_prefilter(report: Dict[str, Any]) -> bool:
    haystack = " ".join([
        str(report.get("title", "")),
        str(report.get("description", "")),
        str(report.get("full_text", "")),
    ]).lower()

    haystack = re.sub(r"\s+", " ", haystack)

    return any(term in haystack for term in IR_HINTS)


def build_relevance_prompt(report: Dict[str, Any]) -> str:
    return f"""You are screening AI incident reports for relevance to a knowledge base used for assessing information-retrieval-oriented AI systems.

The downstream use case is an AI assistant, chatbot, question answering system, search system, or retrieval-augmented system that accesses, retrieves, ranks, selects, recommends, or surfaces information from a knowledge source for users.

Determine whether this incident is relevant to one or more of these IR-related behaviors:

Primary target behaviors:
- Search / Retrieval
- Document Retrieval
- Semantic Search
- Question Answering over a knowledge source
- Retrieval-Augmented Generation (RAG)
- Knowledge Access Assistant
- Dialogue system using retrieved information
- Ranking of retrieved or candidate information

Secondary but still relevant behaviors:
- Recommendation of information or content
- Personalized ranking or feed curation
- Content selection or prioritization
- Suggestion systems that surface information to users

Important:
- An incident does not need to involve a chatbot explicitly.
- A ranking or recommendation incident can still be relevant if it reveals transferable risk patterns about selecting, prioritizing, or surfacing information.
- Return false only if the incident is clearly unrelated to retrieval, search, ranking, recommendation, question answering, or knowledge access behavior.

Return ONLY valid JSON in this exact format:
{{
  "is_relevant": true,
  "relevance_level": "direct",
  "relevance_reason": "short explanation",
  "matched_modes": ["Search/Retrieval", "Question Answering"],
  "detected_system_behavior": "retrieval assistant"
}}

Rules:
- relevance_level must be one of: "direct", "indirect", "not_relevant"
- "direct" means the incident clearly involves IR-oriented behavior central to retrieval, search, QA, ranking, or knowledge access
- "indirect" means the incident is not a retrieval assistant itself, but still contains transferable risk knowledge for IR-oriented systems
- "not_relevant" means the incident is clearly unrelated to information retrieval, search, ranking, recommendation, question answering, or knowledge access
- if not relevant, set is_relevant to false and relevance_level to "not_relevant"
- matched_modes must be a JSON list of short controlled labels
- use labels such as:
  "Search/Retrieval",
  "Question Answering",
  "Dialogue Management",
  "Ranking",
  "Recommendation Scoring",
  "Knowledge Access Assistant",
  "Retrieval-Augmented Assistant"
- detected_system_behavior should be a short application-level label such as:
  "search engine", "QA system", "retrieval assistant", "chatbot", "recommender system", "ranking system", "content feed"
- do not include markdown
- do not include comments
- do not include any text before or after the JSON

INCIDENT:
Report ID: {report["report_id"]}
Title: {report["title"]}
Description: {report["description"]}
Text:
{report["full_text"]}
""".strip()


def stage0_relevance_gate(report: Dict[str, Any]) -> Dict[str, Any]:
    if not cheap_ir_prefilter(report):
        return {
            "is_relevant": False,
            "relevance_level": "not_relevant",
            "relevance_reason": "Keyword prefilter found no IR-related signal",
            "matched_modes": [],
            "detected_system_behavior": "unknown"
        }

    prompt = build_relevance_prompt(report)
    parsed = llm_json("stage0_relevance", prompt, report["report_id"])

    if not parsed or not isinstance(parsed, dict) or parsed.get("_parse_error"):
        return {
            "is_relevant": False,
            "relevance_level": "not_relevant",
            "relevance_reason": "LLM response could not be parsed",
            "matched_modes": [],
            "detected_system_behavior": "unknown"
        }

    matched_modes = parsed.get("matched_modes", [])
    if not isinstance(matched_modes, list):
        matched_modes = []

    return {
        "is_relevant": bool(parsed.get("is_relevant", False)),
        "relevance_level": str(parsed.get("relevance_level", "not_relevant")),
        "relevance_reason": str(parsed.get("relevance_reason", "")),
        "matched_modes": [str(x) for x in matched_modes],
        "detected_system_behavior": str(parsed.get("detected_system_behavior", "unknown"))
    }

## 6. Template-based extraction prompt

In [ ]:
EXTRACTION_TEMPLATE = {
    "meta": {
        "report_id": "",
        "title": "",
        "source_domain": "",
        "url": "",
        "is_relevant": True,
        "relevance_reason": ""
    },
    "system_profile": {
        "ai_task": [],
        "data_input": [],
        "model_type": [],
        "system_type": [],
        "domain": "",
        "stakeholders": []
    },
    "risk_assessment": {
        "risk_category": "",
        "risk_type": "",
        "confidence": ""
    },
    "risk_pattern": {
        "mechanism": "",
        "trigger": "",
        "consequence": "",
        "impact": ""
    },
    "evidence": {
        "system_evidence": [],
        "risk_evidence": [],
        "pattern_evidence": []
    }
}

def build_ir_extraction_prompt(report: Dict[str, Any], sys_taxonomy: str, risk_taxonomy: str) -> str:
    schema_str = json.dumps(EXTRACTION_TEMPLATE, ensure_ascii=False, indent=2)

    return f"""You are extracting structured information from an AI incident report.

Your goal is to convert the incident into structured knowledge for an AI Risk Knowledge Base.

This knowledge base is especially interested in Information Retrieval (IR)-related systems, such as search, retrieval, ranking, recommendation, question answering, and dialogue assistants that access or surface information. However, the incident may describe a broader AI system. In that case, extract the closest fitting system profile and risk information from the report without forcing an IR interpretation.

The extraction has two objectives:
1. System profiling: characterize the AI system involved in the incident using the provided system taxonomy.
2. Risk characterization: identify the relevant risk category, specific risk type, and structured risk pattern expressed in the incident.

Use the following system taxonomy as guidance:
{sys_taxonomy}

Use the following risk taxonomy as guidance:
{risk_taxonomy}

Return ONLY valid JSON with EXACTLY this structure:
{schema_str}

General extraction rules:
- Use only information explicitly stated in the report or strongly supported by the text.
- Do not invent technical details that are not grounded in the report.
- Prefer the closest fitting taxonomy label instead of "unknown".
- Use "unknown" only when a field truly cannot be inferred.
- Use short, normalized phrases for structured fields.
- Reuse taxonomy labels exactly when available.
- Do not create new taxonomy labels when an existing one is close enough.
- All list fields must be valid JSON arrays of strings.
- Evidence fields must contain short verbatim snippets copied directly from the report text.
- Do not paraphrase evidence.
- Do not include markdown.
- Do not include comments.
- Do not include any text before or after the JSON.

System profile rules:
- system_profile must describe the AI system behavior involved in the incident, not just the general topic of the article.
- Extract the closest fitting system behavior from the taxonomy.
- If the incident clearly involves retrieval, ranking, recommendation, question answering, or dialogue over information, capture that explicitly.
- If the incident is not clearly IR-related, still extract the broader AI system profile rather than forcing IR-specific labels.
- Only fill fields that are explicitly supported or strongly implied by the report.

AI Task rules:
- ai_task should describe what the system functionally does.
- Use the closest task labels from the provided taxonomy.
- Possible relevant labels include:
  "Search/Retrieval"
  "Ranking"
  "Recommendation Scoring"
  "Similarity or Matching"
  "Question Answering"
  "Dialogue Management"
  "Tool-using Assistant"
  "Information Extraction"
  "Content Creation"
  "Content Transformation"
  "Classification and labeling"
  "Identification"
  "Anomaly or Outlier Detection"
  "Outcome Prediction"
- ai_task may contain multiple values only if the report clearly supports multiple tasks.
- Do not force IR-related task labels if another task label fits better.

Input Data rules:
- input_data should describe the type of data the system uses, processes, retrieves, ranks, generates over, or learns from.
- Only fill a subfield if it is explicitly stated or strongly implied.
- data_source_type refers to how the data was collected, such as:
  "Collected by Humans"
  "Collected by Sensors"
  "Collected by Human and Sensors"
- data_provenance refers to where the data comes from, such as:
  "Expert Input"
  "Provided Data"
  "Observed Data"
  "Synthetic Data"
  "Derived Data"
- data_structure refers to the structure of the data, such as:
  "Unstructured Data"
  "Semi-structured Data"
  "Structured Data"
  "Complex Structured Data"
- data_dynamics refers to how often data changes, such as:
  "Static Data"
  "Data Updated time-to-time"
  "Data Updated real-time"
- data_identifiability refers to whether the data is personally identifying, such as:
  "Identified Data"
  "Pseudonymised Data"
  "Unlinked Pseudonymised Data"
  "Anonymised Data"
  "Aggregated Data"
- data_rights refers to legal or ownership characteristics, such as:
  "Proprietary Data"
  "Public Data"
  "Personal Data"
- Leave list fields empty if the report does not support them.

Model Objective rules:
- model_objective should describe the AI model or modeling logic involved in the incident.
- Only extract what is supported by the report.
- objective_type should use the closest taxonomy labels, such as:
  "Discriminative Model"
  "Generative Model"
  "Combining Generative and Discriminative Model"
- model_composition should indicate whether the system uses:
  "Model Ensembles"
  "System is Composed of Single AI Model"
- knowledge_acquisition should describe how the model obtains knowledge, using the closest taxonomy label.
- learning_setting should only be filled if the report supports:
  "Centralised Learning"
  "Federated Learning"
- model_maintenance should only be filled if inferable from the report, using:
  "Universal Model"
  "Customisable Model"
  "Tailored Model"
- reasoning_style should reflect whether the system is:
  "Deterministic Model"
  "Probabilistic Model"
  "Combine Deterministic and Probabilistic"
- If the report does not support these details, use empty lists rather than guessing.

System / Operational rules:
- system_operational should describe how the system is deployed or used in practice.
- system_type should be a concise application-level label, such as:
  "search engine"
  "search assistant"
  "retrieval assistant"
  "QA system"
  "chatbot"
  "recommender system"
  "content ranking system"
  "content feed"
  "knowledge assistant"
  "classification system"
  "prediction system"
  "monitoring system"
  "decision-support system"
- Choose the closest fitting label supported by the report.
- autonomy_level should reflect the closest fitting level only if inferable.
- output_actionability should use:
  "advisory"
  "decision-support"
  "actuation"
  only when supported by the report.
- interaction_mode should describe how users or downstream systems interact with the system, using short labels such as:
  "conversational"
  "search query"
  "ranked list"
  "recommendation interface"
  "feed interface"
  "API-based"
  "back-end automation"
- human_role should capture whether humans review, oversee, or rely on the outputs, using short labels such as:
  "human-in-the-loop"
  "human-on-the-loop"
  "human-out-of-the-loop"
  "human consumer of output"
- Do not infer operational details unless there is enough textual support.

Context rules:
- context should describe the application setting in which the system operates.
- domain should be a short application area, such as:
  "search engine"
  "social media"
  "healthcare"
  "education"
  "hiring"
  "finance"
  "media"
  "public service"
  "e-commerce"
  "security"
  "transport"
- purpose should describe what the system is intended to achieve for users or operators.
- stakeholders must be a JSON list of relevant actors affected by the system, such as users, developers, operators, organizations, decision subjects, content creators, or the public.

Risk assessment rules:
- risk_category must align with the provided risk taxonomy and must be one high-level category.
- risk_type must be more specific than risk_category and should capture the closest concrete risk expressed in the incident.
- Prefer the most concrete risk type that is supported by the report.
- confidence must be one of:
  "high"
  "medium"
  "low"
- Use "high" when the classification is clearly supported by the text.
- Use "medium" when the classification is plausible but partly inferred.
- Use "low" when the classification is weakly supported or uncertain.

Risk pattern rules:
- mechanism explains how the risk arises within the system, pipeline, model behavior, interface logic, data handling, optimization logic, or socio-technical setting.
- trigger explains the condition, event, context, or input under which the mechanism is activated.
- consequence explains what the system does wrong or fails to do.
- impact explains who is harmed and how.
- Keep these four fields concise and distinct.
- Do not merge them into one statement.
- If the incident involves IR-related behavior, reflect that in the mechanism or consequence when supported.
- If the incident is not clearly IR-related, describe the broader AI failure pattern instead.

Evidence rules:
- system_evidence must justify the extracted system profile.
- risk_evidence must justify the selected risk category and risk type.
- pattern_evidence must justify the extracted mechanism, trigger, consequence, or impact.
- Evidence must be short verbatim snippets copied directly from the report.
- Prefer 1 to 5 short snippets per evidence field.
- Do not include explanations inside the evidence strings.

Relevance rules:
- Set is_relevant to true if the incident is useful for understanding risks in AI systems, especially when it may inform retrieval, ranking, recommendation, question answering, or knowledge access systems.
- This includes directly relevant IR systems and adjacent systems with transferable risk patterns.
- Set is_relevant to false only if the incident is clearly unrelated to AI system behavior or AI risk assessment.
- relevance_reason must be a short explanation.

Output quality rules:
- Fill every field in the JSON structure.
- Use empty lists for list fields when nothing can be inferred.
- Use "unknown" for string fields that truly cannot be inferred.
- Keep values concise and normalized.
- Ensure the JSON is syntactically valid.

INCIDENT:
Report ID: {report["report_id"]}
Title: {report["title"]}
Description: {report["description"]}
Source Domain: {report["source_domain"]}
URL: {report["url"]}

Text:
{report["full_text"]}
""".strip()

## 7. Extraction + normalization

In [ ]:
def _as_list_of_str(value):
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip() and str(v).strip().lower() != "unknown"]
    if isinstance(value, str):
        v = value.strip()
        if v and v.lower() != "unknown":
            return [v]
    return []


def _as_str(value, default="unknown"):
    if value is None:
        return default
    s = str(value).strip()
    return s if s else default


def normalize_extraction(obj: Dict[str, Any], report: Dict[str, Any], relevance: Dict[str, Any]) -> Dict[str, Any]:
    obj = obj or {}

    meta = obj.get("meta", {}) if isinstance(obj.get("meta"), dict) else {}
    system_profile = obj.get("system_profile", {}) if isinstance(obj.get("system_profile"), dict) else {}
    risk_assessment = obj.get("risk_assessment", {}) if isinstance(obj.get("risk_assessment"), dict) else {}
    risk_pattern = obj.get("risk_pattern", {}) if isinstance(obj.get("risk_pattern"), dict) else {}
    evidence = obj.get("evidence", {}) if isinstance(obj.get("evidence"), dict) else {}

    return {
        "meta": {
            "report_id": _as_str(meta.get("report_id", report.get("report_id", "")), ""),
            "title": _as_str(meta.get("title", report.get("title", "")), ""),
            "source_domain": _as_str(meta.get("source_domain", report.get("source_domain", "")), ""),
            "url": _as_str(meta.get("url", report.get("url", "")), ""),
            "is_relevant": bool(relevance.get("is_relevant", True)),
            "relevance_reason": _as_str(relevance.get("relevance_reason", ""), "")
        },
        "system_profile": {
            "ai_task": _as_list_of_str(system_profile.get("ai_task", [])),
            "data_input": _as_list_of_str(system_profile.get("data_input", [])),
            "model_type": _as_list_of_str(system_profile.get("model_type", [])),
            "system_type": _as_list_of_str(system_profile.get("system_type", [])),
            "domain": _as_str(system_profile.get("domain", "unknown")),
            "stakeholders": _as_list_of_str(system_profile.get("stakeholders", []))
        },
        "risk_assessment": {
            "risk_category": _as_str(risk_assessment.get("risk_category", "unknown")),
            "risk_type": _as_str(risk_assessment.get("risk_type", "unknown")),
            "confidence": _as_str(risk_assessment.get("confidence", "low"))
        },
        "risk_pattern": {
            "mechanism": _as_str(risk_pattern.get("mechanism", "unknown")),
            "trigger": _as_str(risk_pattern.get("trigger", "unknown")),
            "consequence": _as_str(risk_pattern.get("consequence", "unknown")),
            "impact": _as_str(risk_pattern.get("impact", "unknown"))
        },
        "evidence": {
            "system_evidence": _as_list_of_str(evidence.get("system_evidence", [])),
            "risk_evidence": _as_list_of_str(evidence.get("risk_evidence", [])),
            "pattern_evidence": _as_list_of_str(evidence.get("pattern_evidence", []))
        },
        "matched_modes": _as_list_of_str(relevance.get("matched_modes", [])),
        "relevance_level": _as_str(relevance.get("relevance_level", "unknown")),
        "detected_system_behavior": _as_str(relevance.get("detected_system_behavior", "unknown"))
    }


def stage1_extract(report: Dict[str, Any], relevance: Dict[str, Any]) -> Dict[str, Any]:
    prompt = build_ir_extraction_prompt(report, ai_sys_taxonomy_text, ai_risk_taxonomy_text)
    parsed = llm_json("stage1_extract", prompt, report["report_id"])

    if not parsed or not isinstance(parsed, dict) or parsed.get("_parse_error"):
        parsed = {}

    return normalize_extraction(parsed, report, relevance)


def process_report(report: Dict[str, Any]) -> Dict[str, Any]:
    relevance = stage0_relevance_gate(report)

    if not relevance["is_relevant"]:
        return {
            "meta": {
                "report_id": report["report_id"],
                "title": report["title"],
                "source_domain": report["source_domain"],
                "url": report["url"],
                "is_relevant": False,
                "relevance_reason": relevance["relevance_reason"],
            },
            "system_profile": {
                "ai_task": [],
                "data_input": [],
                "model_type": [],
                "system_type": [],
                "domain": "unknown",
                "stakeholders": []
            },
            "risk_assessment": {
                "risk_category": "unknown",
                "risk_type": "unknown",
                "confidence": "low"
            },
            "risk_pattern": {
                "mechanism": "unknown",
                "trigger": "unknown",
                "consequence": "unknown",
                "impact": "unknown"
            },
            "evidence": {
                "system_evidence": [],
                "risk_evidence": [],
                "pattern_evidence": []
            },
            "matched_modes": relevance.get("matched_modes", []),
            "relevance_level": relevance.get("relevance_level", "not_relevant"),
            "detected_system_behavior": relevance.get("detected_system_behavior", "unknown")
        }

    return stage1_extract(report, relevance)

## 8. Run the pipeline

In [9]:

normalized_reports = [normalize_report_row(row) for _, row in reports_df.iterrows()]

if CONFIG["MAX_ROWS"] is not None:
    normalized_reports = normalized_reports[:CONFIG["MAX_ROWS"]]

print(f"Processing {len(normalized_reports)} reports...")

results = []
for idx, report in enumerate(normalized_reports, start=1):
    relevance = {
        "is_relevant": False,
        "relevance_reason": "unknown",
        "relevance_level": "unknown",
        "matched_modes": [],
        "detected_system_behavior": "unknown"
    }

    try:
        logger.info(f"[{idx}/{len(normalized_reports)}] Processing report {report['report_id']}")

        relevance = stage0_relevance_gate(report)

        if not relevance["is_relevant"]:
            results.append({
                "meta": {
                    "report_id": report["report_id"],
                    "title": report["title"],
                    "source_domain": report["source_domain"],
                    "url": report["url"],
                    "is_relevant": False,
                    "relevance_reason": relevance["relevance_reason"],
                },
                "system_profile": {},
                "risk_assessment": {},
                "risk_pattern": {},
                "evidence": {},
                "matched_modes": relevance.get("matched_modes", []),
                "relevance_level": relevance.get("relevance_level", "not_relevant"),
                "detected_system_behavior": relevance.get("detected_system_behavior", "unknown")
            })
            continue

        results.append(stage1_extract(report, relevance))

    except Exception as exc:
        logger.exception(f"Failed on report {report['report_id']}: {exc}")
        results.append({
            "meta": {
                "report_id": report["report_id"],
                "title": report["title"],
                "source_domain": report["source_domain"],
                "url": report["url"],
                "is_relevant": False,
                "relevance_reason": f"pipeline_error: {exc}",
            },
            "system_profile": {},
            "risk_assessment": {},
            "risk_pattern": {},
            "evidence": {},
            "matched_modes": relevance.get("matched_modes", []),
            "relevance_level": relevance.get("relevance_level", "unknown"),
            "detected_system_behavior": relevance.get("detected_system_behavior", "unknown")
        })

print(f"Finished: {len(results)} results")

2026-04-08 04:58:01,451 - __main__ - INFO - [1/100] Processing report 5d34b8c29ced494f010ed45a
2026-04-08 04:58:01,453 - __main__ - INFO - [2/100] Processing report 5d34b8c29ced494f010ed461


Processing 100 reports...


2026-04-08 04:58:11,393 - __main__ - INFO - [3/100] Processing report 5d34b8c29ced494f010ed464
2026-04-08 04:58:15,982 - __main__ - INFO - [4/100] Processing report 5d34b8c29ced494f010ed45b
2026-04-08 04:58:21,782 - __main__ - INFO - [5/100] Processing report 5d34b8c29ced494f010ed462
2026-04-08 04:58:28,280 - __main__ - INFO - [6/100] Processing report 5d34b8c29ced494f010ed460
2026-04-08 04:58:34,967 - __main__ - INFO - [7/100] Processing report 5d34b8c29ced494f010ed465
2026-04-08 04:58:41,180 - __main__ - INFO - [8/100] Processing report 5d34b8c29ced494f010ed467
2026-04-08 04:58:48,239 - __main__ - INFO - [9/100] Processing report 5d34b8c29ced494f010ed45c
2026-04-08 04:58:51,322 - __main__ - INFO - [10/100] Processing report 5d34b8c29ced494f010ed463
2026-04-08 04:58:57,149 - __main__ - INFO - [11/100] Processing report 5d34b8c29ced494f010ed45e
2026-04-08 04:58:59,396 - __main__ - INFO - [12/100] Processing report 5d34b8c29ced494f010ed45d
2026-04-08 04:59:05,443 - __main__ - INFO - [13

Finished: 100 results


## 9. Save results

In [12]:

output_path = Path(CONFIG["OUTPUT_JSON"])
output_path.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"Saved extraction output to: {output_path}")
print(f"Relevant cases: {sum(1 for r in results if r.get('meta', {}).get('is_relevant') is True)}")


Saved extraction output to: D:\Dev\airiskkg\data\outputs\output_reprots.json
Relevant cases: 60


## 10. Quick inspection

In [14]:

relevant_only = [r for r in results if r.get("meta", {}).get("is_relevant") is True]
pd.DataFrame([
    {
        "report_id": r["meta"]["report_id"],
        "title": r["meta"]["title"],
        "ai_task": r.get("system_profile", {}).get("ai_task", ""),
        "risk_category": r.get("risk_assessment", {}).get("risk_category", ""),
        "risk_type": r.get("risk_assessment", {}).get("risk_type", ""),
    }
    for r in relevant_only[:20]
])


,report_id,title,ai_task,risk_category,risk_type
0,5d34b8c29ced494f010ed461,"YouTube Kids, Criticized for Content, Introduc...",[],unknown,unknown
1,5d34b8c29ced494f010ed464,Children's YouTube is still churning out blood...,[],unknown,unknown
2,5d34b8c29ced494f010ed45b,YouTube Kids app is STILL showing disturbing v...,[],unknown,unknown
3,5d34b8c29ced494f010ed462,YouTube suggested conspiracy videos to childre...,[],unknown,unknown
4,5d34b8c29ced494f010ed460,YouTube to crack down on inappropriate content...,[],unknown,unknown
5,5d34b8c29ced494f010ed465,YouTube says it will crack down on bizarre vid...,[],unknown,unknown
6,5d34b8c29ced494f010ed467,YouTube Kids Is Nowhere Near as Innocent As It...,[],unknown,unknown
7,5d34b8c29ced494f010ed45c,Disturbing YouTube Kids video shows Mickey Mou...,[],unknown,unknown
8,5d34b8c29ced494f010ed463,Remove YouTube Kids app until it eliminates it...,[],unknown,unknown
9,5d34b8c29ced494f010ed45d,YouTube has thousands of disturbing videos tar...,[],unknown,unknown
